# Module 3 • Classical Natural Language Processing

# Lesson 13 • Stemming and Lemmatization in Practice

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Beginner  
**Estimated study time:** 90–120 minutes

---

## Scope

This lesson applies stemming and lemmatization to practical NLP pipelines.
It compares their effects on vocabulary, search, classification, error types,
multilingual processing, and Arabic text.

## Learning Objectives

After completing this lesson, the learner should be able to:

- distinguish stemming from lemmatization;
- explain when normalization may help classical NLP models;
- apply English stemming with standard algorithms;
- explain why lemmatization may require part-of-speech information;
- identify overstemming and understemming;
- compare vocabulary before and after normalization;
- build retrieval and classification pipelines with normalization;
- avoid leakage by fitting transformations inside a pipeline;
- explain why Arabic stemming and root extraction require special care;
- evaluate normalization using both linguistic errors and downstream metrics.

## Table of Contents

1. Why Normalize Word Forms?
2. Stemming
3. Lemmatization
4. Stemming Versus Lemmatization
5. English Stemming in Practice
6. Overstemming and Understemming
7. Lemmatization and Part of Speech
8. Vocabulary Reduction
9. Search and Retrieval
10. Classification Pipelines
11. Avoiding Data Leakage
12. Configurable Normalization
13. Arabic Stemming and Root Extraction
14. Multilingual Considerations
15. Evaluation and Error Analysis
16. Testing and Reproducibility
17. Knowledge Check
18. Exercises
19. Summary and Next Lesson

# 1. Why Normalize Word Forms?

One lexical idea may appear in several surface forms:

```text
connect
connects
connected
connecting
connection
connections
```

A sparse classical model may treat each form as a separate feature.
Normalization attempts to reduce selected variation.

In [ ]:
from collections import Counter

forms = [
    "connect",
    "connects",
    "connected",
    "connecting",
    "connection",
    "connections",
]

counts = Counter(forms)

print("Surface vocabulary:", sorted(counts))
print("Vocabulary size:", len(counts))

Normalization may improve:

- search recall;
- feature sharing;
- vocabulary compactness;
- robustness to inflection;
- classical text classification.

It may also merge distinctions that matter to the task.

# 2. Stemming

**Stemming** applies heuristic rules to remove or replace affixes.

Example:

```text
studies   → studi
studying  → studi
relational → relat
```

A stem does not have to be a valid dictionary word.

In [ ]:
def educational_stem(word: str) -> str:
    """Small educational stemmer, not a production algorithm."""
    word = word.lower()

    replacements = [
        ("ies", "y"),
        ("ing", ""),
        ("edly", ""),
        ("ed", ""),
        ("es", ""),
        ("s", ""),
    ]

    for suffix, replacement in replacements:
        if word.endswith(suffix) and len(word) > len(suffix) + 2:
            return word[:-len(suffix)] + replacement

    return word


for word in ["studies", "studying", "walked", "boxes", "cats", "analysis"]:
    print(f"{word:<10} -> {educational_stem(word)}")

The final example shows why simple suffix deletion is unsafe: the final `s` in
`analysis` is not a plural suffix.

# 3. Lemmatization

**Lemmatization** maps a word form to a dictionary form called a **lemma**.

```text
am, is, are → be
went → go
mice → mouse
better → good
```

Accurate lemmatization may require:

- part-of-speech information;
- morphological features;
- a lexicon;
- context.

In [ ]:
irregular_lemmas = {
    ("went", "VERB"): "go",
    ("was", "VERB"): "be",
    ("were", "VERB"): "be",
    ("mice", "NOUN"): "mouse",
    ("children", "NOUN"): "child",
    ("better", "ADJ"): "good",
}

def educational_lemmatize(word: str, part_of_speech: str) -> str:
    key = (word.lower(), part_of_speech.upper())

    if key in irregular_lemmas:
        return irregular_lemmas[key]

    lowered = word.lower()
    pos = part_of_speech.upper()

    if pos == "NOUN" and lowered.endswith("ies") and len(lowered) > 4:
        return lowered[:-3] + "y"

    if pos == "NOUN" and lowered.endswith("s") and not lowered.endswith("ss"):
        return lowered[:-1]

    if pos == "VERB" and lowered.endswith("ing") and len(lowered) > 5:
        return lowered[:-3]

    if pos == "VERB" and lowered.endswith("ed") and len(lowered) > 4:
        return lowered[:-2]

    return lowered

In [ ]:
examples = [
    ("went", "VERB"),
    ("mice", "NOUN"),
    ("studies", "NOUN"),
    ("walking", "VERB"),
    ("better", "ADJ"),
]

for word, pos in examples:
    lemma = educational_lemmatize(word, pos)
    print(f"{word:<10} {pos:<5} -> {lemma}")

This function is educational. Production lemmatizers require more complete
linguistic knowledge or trained models.

# 4. Stemming Versus Lemmatization

| Property | Stemming | Lemmatization |
|---|---|---|
| Main method | heuristic affix processing | linguistic analysis |
| Output | stem-like string | dictionary lemma |
| Context needed | often no | frequently yes |
| Speed | usually fast | often more expensive |
| Linguistic validity | not guaranteed | intended |
| Typical use | retrieval and sparse baselines | linguistically informed pipelines |

Example:

```text
better
```

A stemmer may return a reduced string, while a lemmatizer with adjective
information may return `good`.

# 5. English Stemming in Practice

NLTK provides several stemming algorithms. The notebook uses them when the
package is available and falls back to the educational stemmer otherwise.

In [ ]:
try:
    from nltk.stem import LancasterStemmer, PorterStemmer, SnowballStemmer

    porter = PorterStemmer()
    lancaster = LancasterStemmer()
    snowball = SnowballStemmer("english")
    NLTK_STEMMERS_AVAILABLE = True
except Exception:
    NLTK_STEMMERS_AVAILABLE = False

print("NLTK stemmers available:", NLTK_STEMMERS_AVAILABLE)

In [ ]:
stem_words = [
    "connect",
    "connected",
    "connecting",
    "connection",
    "relational",
    "generously",
    "studies",
]

rows = []

for word in stem_words:
    if NLTK_STEMMERS_AVAILABLE:
        porter_result = porter.stem(word)
        lancaster_result = lancaster.stem(word)
        snowball_result = snowball.stem(word)
    else:
        porter_result = educational_stem(word)
        lancaster_result = educational_stem(word)
        snowball_result = educational_stem(word)

    rows.append(
        (
            word,
            porter_result,
            lancaster_result,
            snowball_result,
        )
    )

import pandas as pd

pd.DataFrame(
    rows,
    columns=["Word", "Porter", "Lancaster", "Snowball"],
)

Different stemmers use different rule sets and aggressiveness. A more
aggressive stemmer may reduce vocabulary further while increasing erroneous
merges.

# 6. Overstemming and Understemming

**Overstemming** merges words that should remain distinct.

Example:

```text
policy
police
```

An overly aggressive stemmer may map them too closely.

**Understemming** fails to merge related forms.

Example:

```text
analysis
analyze
analytical
```

In [ ]:
error_pairs = pd.DataFrame(
    [
        ("policy", "police", "possible overstemming risk"),
        ("universe", "university", "possible overstemming risk"),
        ("analysis", "analyze", "possible understemming"),
        ("go", "went", "irregular forms may remain separate"),
    ],
    columns=["Word 1", "Word 2", "Issue"],
)

error_pairs

Error analysis should inspect whether normalization preserves distinctions
needed by the task.

# 7. Lemmatization and Part of Speech

The same surface form may require different lemmas in different contexts.

```text
The leaves fall.   → leaves = leaf, noun
She leaves early.  → leaves = leave, verb
```

In [ ]:
pos_sensitive_examples = pd.DataFrame(
    [
        ("The leaves fall.", "leaves", "NOUN", "leaf"),
        ("She leaves early.", "leaves", "VERB", "leave"),
        ("The record is complete.", "record", "NOUN", "record"),
        ("They record the meeting.", "record", "VERB", "record"),
    ],
    columns=["Sentence", "Surface form", "POS", "Lemma"],
)

pos_sensitive_examples

In [ ]:
for word, pos in [("leaves", "NOUN"), ("leaves", "VERB")]:
    print(
        f"{word:<8} as {pos:<5} -> "
        f"{educational_lemmatize(word, pos)}"
    )

A lemmatizer without reliable part-of-speech information may select the wrong
lemma or fail to normalize the word.

# 8. Vocabulary Reduction

Normalization can reduce the number of distinct features in a corpus.

In [ ]:
documents = [
    "the student connects the device",
    "the devices are connected",
    "the system is connecting",
    "several connections failed",
]

def whitespace_tokens(text: str) -> list[str]:
    return text.lower().split()

def stem_document(text: str) -> list[str]:
    tokens = whitespace_tokens(text)

    if NLTK_STEMMERS_AVAILABLE:
        return [porter.stem(token) for token in tokens]

    return [educational_stem(token) for token in tokens]

raw_vocabulary = {
    token
    for document in documents
    for token in whitespace_tokens(document)
}

stemmed_vocabulary = {
    token
    for document in documents
    for token in stem_document(document)
}

print("Raw vocabulary size:", len(raw_vocabulary))
print("Stemmed vocabulary size:", len(stemmed_vocabulary))
print()
print("Raw vocabulary:", sorted(raw_vocabulary))
print("Stemmed vocabulary:", sorted(stemmed_vocabulary))

A smaller vocabulary may improve feature sharing, but vocabulary reduction is
not automatically beneficial.

# 9. Search and Retrieval

Normalization may improve recall when a query and document use different word
forms.

In [ ]:
retrieval_documents = [
    "The system connects to the server.",
    "The devices are connected securely.",
    "Connection quality is stable.",
    "The cable was disconnected.",
    "The model predicts sentiment.",
]

query = "connect"

literal_matches = [
    document
    for document in retrieval_documents
    if query in document.lower().split()
]

query_stem = stem_document(query)[0]

stem_matches = []

for document in retrieval_documents:
    document_stems = stem_document(document.replace(".", ""))
    if query_stem in document_stems:
        stem_matches.append(document)

print("Literal matches:")
for document in literal_matches:
    print("-", document)

print("\nStem-based matches:")
for document in stem_matches:
    print("-", document)

Stem-based matching may improve recall but can reduce precision if unrelated
words receive the same stem.

# 10. Classification Pipelines

Classical classifiers often use Bag-of-Words or TF-IDF features. Stemming or
lemmatization can be applied before vectorization.

In [ ]:
training_texts = [
    "excellent service and helpful staff",
    "the service was excellent",
    "helpful response and fast support",
    "terrible service and rude staff",
    "the response was terrible",
    "slow support and unhelpful staff",
    "excellent and friendly assistance",
    "rude and unhelpful service",
    "support responded quickly",
    "service responded slowly",
]

training_labels = [
    "positive",
    "positive",
    "positive",
    "negative",
    "negative",
    "negative",
    "positive",
    "negative",
    "positive",
    "negative",
]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

raw_pipeline = Pipeline(
    [
        ("tfidf", TfidfVectorizer()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

raw_scores = cross_val_score(
    raw_pipeline,
    training_texts,
    training_labels,
    cv=5,
    scoring="f1_macro",
)

print("Raw-text macro F1 scores:", raw_scores.round(3))
print("Mean macro F1:", raw_scores.mean().round(3))

## 10.1 Stemmed Analyzer

`TfidfVectorizer` can receive a custom analyzer that returns normalized tokens.

In [ ]:
def stemmed_analyzer(text: str) -> list[str]:
    cleaned = text.lower().replace(".", "")
    return stem_document(cleaned)

stemmed_pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(analyzer=stemmed_analyzer),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

stemmed_scores = cross_val_score(
    stemmed_pipeline,
    training_texts,
    training_labels,
    cv=5,
    scoring="f1_macro",
)

print("Stemmed macro F1 scores:", stemmed_scores.round(3))
print("Mean macro F1:", stemmed_scores.mean().round(3))

The dataset is deliberately small, so the scores are only a workflow
demonstration. A real experiment requires more data, repeated validation, and
error analysis.

In [ ]:
comparison = pd.DataFrame(
    {
        "Configuration": ["Raw text", "Stemmed text"],
        "Mean macro F1": [
            raw_scores.mean(),
            stemmed_scores.mean(),
        ],
    }
)

comparison

# 11. Avoiding Data Leakage

Text normalization rules that learn from data should be fitted only on the
training partition.

Examples include:

- vocabulary selection;
- spelling dictionaries built from the full dataset;
- frequency-based stop-word lists;
- learned segmentation models;
- feature selection.

Scikit-learn pipelines help ensure that learned steps are fitted inside each
training fold.

Deterministic stemming rules do not learn from the dataset, but vectorizer
vocabularies do. They should remain inside the pipeline.

# 12. Configurable Normalization

A reusable normalizer should make the selected method explicit.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class WordNormalizationConfig:
    method: str = "none"
    part_of_speech: str = "NOUN"


def normalize_word(
    word: str,
    config: WordNormalizationConfig,
) -> str:
    method = config.method.lower()

    if method == "none":
        return word.lower()

    if method == "stem":
        if NLTK_STEMMERS_AVAILABLE:
            return porter.stem(word.lower())
        return educational_stem(word)

    if method == "lemma":
        return educational_lemmatize(
            word,
            config.part_of_speech,
        )

    raise ValueError(
        "method must be 'none', 'stem', or 'lemma'"
    )

In [ ]:
word = "studies"

for config in [
    WordNormalizationConfig(method="none"),
    WordNormalizationConfig(method="stem"),
    WordNormalizationConfig(
        method="lemma",
        part_of_speech="NOUN",
    ),
]:
    print(config, "->", normalize_word(word, config))

Configuration values should be stored with the experiment and model artifacts.

# 13. Arabic Stemming and Root Extraction

Arabic normalization is more complex because words may include:

- conjunctions;
- prepositions;
- the definite article;
- prefixes and suffixes;
- attached pronouns;
- root-and-pattern morphology;
- omitted short vowels;
- dialectal variation.

Arabic approaches may include:

- **light stemming:** remove selected clitics and affixes;
- **root extraction:** attempt to identify a consonantal root;
- **lemmatization:** return a dictionary form;
- **morphological analysis:** return segmentation and features.

Root extraction is not equivalent to lemmatization.

In [ ]:
ARABIC_PREFIXES = ["وال", "بال", "كال", "فال", "لل", "ال", "و", "ف", "ب", "ك", "ل"]
ARABIC_SUFFIXES = ["كما", "هما", "كم", "كن", "هم", "هن", "ها", "نا", "ات", "ون", "ين", "ة", "ي"]

def educational_arabic_light_stem(word: str) -> str:
    """Small educational light stemmer, not a production implementation."""
    result = word

    for prefix in sorted(ARABIC_PREFIXES, key=len, reverse=True):
        if result.startswith(prefix) and len(result) > len(prefix) + 2:
            result = result[len(prefix):]
            break

    for suffix in sorted(ARABIC_SUFFIXES, key=len, reverse=True):
        if result.endswith(suffix) and len(result) > len(suffix) + 2:
            result = result[:-len(suffix)]
            break

    return result


arabic_words = [
    "والكتاب",
    "بالمدرسة",
    "المعلمات",
    "سيكتبون",
    "كتابها",
]

for word in arabic_words:
    print(f"{word:<12} -> {educational_arabic_light_stem(word)}")

Surface rules can over-strip lexical letters or fail to identify internal
patterns. Arabic stemming should be evaluated with native-speaker review and
task-specific data.

## 13.1 Arabic Search Trade-Off

Light stemming may improve recall by merging attached forms, but aggressive
root extraction may merge semantically distinct words sharing a root.

In [ ]:
arabic_tradeoffs = pd.DataFrame(
    [
        ("والكتاب", "كتاب", "useful clitic removal"),
        ("مكتبة", "كتب", "root extraction may be too aggressive for some tasks"),
        ("كاتب", "كتب", "shared root, different lexical meaning"),
        ("كتاب", "كتب", "shared root, different word form"),
    ],
    columns=["Surface form", "Possible normalized form", "Observation"],
)

arabic_tradeoffs

# 14. Multilingual Considerations

A normalization method should match:

- language;
- writing system;
- morphology;
- domain;
- tokenization scheme;
- task.

An English suffix stemmer should not be applied to Arabic, French, Turkish, or
another language without appropriate adaptation.

In [ ]:
multilingual_policy = pd.DataFrame(
    [
        ("English", "Porter or Snowball stemming; POS-aware lemmatization"),
        ("Arabic", "light stemming or morphological analysis"),
        ("Turkish", "morphology-aware segmentation or analysis"),
        ("French", "language-specific stemming or lemmatization"),
        ("Multilingual corpus", "language identification plus language-specific processing"),
    ],
    columns=["Language setting", "Possible strategy"],
)

multilingual_policy

# 15. Evaluation and Error Analysis

Normalization may be evaluated through:

- vocabulary reduction;
- retrieval precision and recall;
- classification macro F1;
- exact lemma accuracy;
- stemming error analysis;
- performance on rare forms;
- downstream robustness;
- analysis by language and domain.

## 15.1 Common Error Categories

- overstemming;
- understemming;
- invalid stem;
- incorrect lemma;
- wrong part of speech;
- irregular form not normalized;
- named entity damaged;
- technical term altered;
- language mismatch;
- Arabic prefix or suffix over-stripped;
- semantically distinct words merged;
- inconsistent training and deployment normalization.

In [ ]:
normalization_errors = pd.DataFrame(
    [
        ("analysis", "analysi", "invalid or unhelpful stem"),
        ("better", "better", "irregular lemma missed"),
        ("Apple", "appl", "named entity damaged"),
        ("policy / police", "same stem", "overstemming"),
        ("كتاب / كاتب", "كتب", "distinct Arabic words merged"),
    ],
    columns=["Input", "Output", "Error type"],
)

normalization_errors

# 16. Testing and Reproducibility

Unit tests should cover:

- already normalized words;
- inflected forms;
- irregular forms;
- short words;
- names;
- punctuation;
- multilingual input;
- empty strings;
- method configuration.

In [ ]:
test_cases = [
    ("cats", WordNormalizationConfig("stem"), None),
    ("went", WordNormalizationConfig("lemma", "VERB"), "go"),
    ("mice", WordNormalizationConfig("lemma", "NOUN"), "mouse"),
    ("", WordNormalizationConfig("none"), ""),
]

for word, config, expected in test_cases:
    actual = normalize_word(word, config)
    passed = expected is None or actual == expected
    print(
        f"word={word!r:<8} method={config.method:<6} "
        f"output={actual!r:<10} passed={passed}"
    )

Record the algorithm name, language, version, custom rules, and stop-word
policy with each experiment.

# 17. Knowledge Check

1. What is stemming?
2. What is lemmatization?
3. Why may a stem be an invalid word?
4. Why does lemmatization often require part-of-speech information?
5. What is overstemming?
6. What is understemming?
7. How can normalization reduce vocabulary size?
8. How can stemming improve retrieval recall?
9. Why may stemming reduce retrieval precision?
10. Why should learned preprocessing remain inside a pipeline?
11. How does Arabic light stemming differ from root extraction?
12. Why is root extraction not the same as lemmatization?
13. Which downstream metrics can evaluate normalization?

# 18. Exercises

## Exercise 1 — Stemmer Comparison

Compare Porter, Lancaster, and Snowball stemming on fifty English words.

## Exercise 2 — Error Analysis

Label each disagreement as overstemming, understemming, invalid stem, or
acceptable variation.

## Exercise 3 — Lemmatization

Extend the educational lemmatizer with twenty irregular English forms.

## Exercise 4 — POS Sensitivity

Create sentence pairs in which the same surface word has different parts of
speech and lemmas.

## Exercise 5 — Vocabulary Study

Measure vocabulary size before and after stemming on a small corpus.

## Exercise 6 — Retrieval

Compare literal, stem-based, and lemma-based retrieval.

## Exercise 7 — Classification

Compare raw, stemmed, and lemmatized TF-IDF classifiers using macro F1.

## Exercise 8 — Arabic Analysis

Apply a light stemmer to Arabic examples and manually identify over-stripping
and under-stripping.

## Challenge Exercises

1. Implement a scikit-learn transformer for configurable stemming.
2. Add part-of-speech tagging before lemmatization.
3. Compare normalization across two languages.
4. Build an audit report showing how every token changed.
5. Test whether stemming improves recall on a domain-specific search task.

# 19. Summary and Next Lesson

In this lesson:

- stemming applied heuristic affix rules;
- lemmatization targeted dictionary forms;
- part-of-speech information improved lemma selection;
- different stemmers produced different levels of reduction;
- overstemming and understemming described common failure modes;
- normalization reduced vocabulary and changed feature sharing;
- stem-based retrieval could improve recall while harming precision;
- TF-IDF pipelines allowed direct downstream comparison;
- learned preprocessing and vectorization remained inside pipelines to avoid
  leakage;
- Arabic light stemming, root extraction, and lemmatization were distinguished;
- evaluation combined linguistic error analysis with downstream metrics.

## Next Lesson

**Lesson 14: Stop Words, N-Grams, and Feature Engineering** examines stop-word
policies, unigram and n-gram features, feature frequency, vocabulary pruning,
and task-aware feature design.

# References

- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.
- Porter, M. F. *An Algorithm for Suffix Stripping*.
- NLTK stemming documentation.
- scikit-learn text-feature extraction documentation.
- Arabic stemming, root extraction, and morphological-analysis literature.